<a href="https://colab.research.google.com/github/edgi-govdata-archiving/GHG-CDP/blob/main/ECHO_GHG_Merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas
import geopandas

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#First, we merge annual CAA violation counts by FRSID with GHG facility metadata

In [2]:
caa_violations = pandas.read_csv("/content/drive/Shareddrives/EDGI - Shared NEW/12_Environmental_Enforcement_Watch/09_Data/Climate_Data_Project/ECHO data by FRSID/ECHO-Cross-Programs_GHGRP.ipynb outputs/ghgrp_fac_caa_violations_2016-2023.csv", dtype = {"REGISTRY_ID": "string"})

In [3]:
caa_violations

,PGM_SYS_ID,ACTIVITY_ID,AGENCY_TYPE_DESC,STATE_CODE,COMP_DETERMINATION_UID,ENF_RESPONSE_POLICY_CODE,PROGRAM_CODES,PROGRAM_DESCS,POLLUTANT_CODES,POLLUTANT_DESCS,...,FAC_DERIVED_CD113,FAC_PERCENT_MINORITY,FAC_POP_DEN,FAC_DERIVED_HUC,FAC_SIC_CODES,FAC_NAICS_CODES,DFR_URL,EARLIEST_FRV_DETERM_DATE,AIR_LCON_CODE,Date
0,MA0000002511700030,3603608015,State,MA,MA000A100720,FRV,CAASIP,State Implementation Plan for National Primary...,10193 300000282,Carbon monoxide NITROGEN OXIDES,...,1.0,14.744,1476.98,1100005.0,2821,325211 488119 333999 551114 531120,http://echo.epa.gov/detailed-facility-report?f...,2023-05-26,NaN,2023-05-26
1,NJ0000003401500059,3603111968,State,NJ,NJ000A92563,FRV,CAASIP CAATVP,State Implementation Plan for National Primary...,NaN,NaN,...,1.0,23.777,957.84,2040202.0,2911 5171,324110 324121 42471,http://echo.epa.gov/detailed-facility-report?f...,2021-06-09,NaN,2021-06-09
2,NJ0000003401500059,3604432299,State,NJ,NJ000A128229,FRV,CAANSPS CAASIP CAATVP,New Source Performance Standards State Impleme...,NaN,NaN,...,1.0,23.777,957.84,2040202.0,2911 5171,324110 324121 42471,http://echo.epa.gov/detailed-facility-report?f...,2023-06-08,NaN,2023-06-08
3,NJ0000003401500059,3601281400,State,NJ,NJ000A70044,FRV,CAASIP CAATVP,State Implementation Plan for National Primary...,NaN,NaN,...,1.0,23.777,957.84,2040202.0,2911 5171,324110 324121 42471,http://echo.epa.gov/detailed-facility-report?f...,2017-11-28,NaN,2017-11-28
4,NJ0000003401500059,3602868021,State,NJ,NJ000A89040,FRV,CAASIP CAATVP,State Implementation Plan for National Primary...,NaN,NaN,...,1.0,23.777,957.84,2040202.0,2911 5171,324110 324121 42471,http://echo.epa.gov/detailed-facility-report?f...,2020-05-19,NaN,2020-05-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10794,IL000155010AAJ,3603779716,State,IL,IL000AA-2023-00185,FRV,CAASIP,State Implementation Plan for National Primary...,10193 300000319 300000320,Carbon monoxide PARTICULATE MATTER < 10 UM PAR...,...,16.0,6.803,46.91,7130001.0,2869 3273 4612,325193 486110,http://echo.epa.gov/detailed-facility-report?f...,2023-08-23,NaN,2023-08-23
10795,IL000155010AAJ,3601261645,State,IL,IL000AA-2017-00137,FRV,CAASIP,State Implementation Plan for National Primary...,300000242 300000243,TOTAL HAZARDOUS AIR POLLUTANTS (HAPS) VOLATILE...,...,16.0,6.803,46.91,7130001.0,2869 3273 4612,325193 486110,http://echo.epa.gov/detailed-facility-report?f...,2017-11-05,NaN,2017-11-05
10796,IL000155010AAJ,3602227629,State,IL,IL000AA-2020-00189,FRV,CAASIP,State Implementation Plan for National Primary...,300000242 300000243 300000329,FACIL TOTAL HAZARDOUS AIR POLLUTANTS (HAPS) VO...,...,16.0,6.803,46.91,7130001.0,2869 3273 4612,325193 486110,http://echo.epa.gov/detailed-facility-report?f...,2020-07-02,NaN,2020-07-02
10797,IL000155010AAJ,3602098607,State,IL,IL000AA-2020-00003,FRV,CAASIP,State Implementation Plan for National Primary...,300000243,VOLATILE ORGANIC COMPOUNDS (VOCS),...,16.0,6.803,46.91,7130001.0,2869 3273 4612,325193 486110,http://echo.epa.gov/detailed-facility-report?f...,2019-12-30,NaN,2019-12-30


In [4]:
caa_violations["YEAR"] = pandas.to_datetime(caa_violations["Date"]).dt.year

violations_fac_year = (
    caa_violations[caa_violations["YEAR"].between(2016, 2023)]
    .groupby(["REGISTRY_ID", "YEAR"], as_index=False)
    .agg(
        ANNUAL_CAA_VIOLATION_COUNT=("ACTIVITY_ID", "count"),
    )
)

In [5]:
violations_fac_year

,REGISTRY_ID,YEAR,ANNUAL_CAA_VIOLATION_COUNT
0,110000308471,2023,1
1,110000308480,2016,1
2,110000308480,2023,1
3,110000308523,2016,1
4,110000308523,2018,1
...,...,...,...
5536,110071349304,2019,27
5537,110071349304,2020,8
5538,110071349304,2021,16
5539,110071349304,2022,27


In [6]:
ghg = pandas.read_csv("https://raw.githubusercontent.com/edgi-govdata-archiving/GHG-CDP/refs/heads/main/output_files/facility_frsid_metadata.csv", dtype = {"FRS Id": "string"})

In [7]:
ghg_columns = ghg.columns.to_list()
ghg_columns

['Facility Id',
 'Facility Name',
 'FRS Id',
 'City',
 'State',
 'Zip Code',
 'Latest Reporting Year',
 'Address',
 'County',
 'Latitude',
 'Longitude',
 'Primary NAICS Code',
 'Industry Type (subparts)',
 'Industry Type (sectors)',
 'Is_Direct_Emitter',
 'Is_Supplier',
 'Basin',
 'State where Emissions Occur',
 'Does the facility employ continuous emissions monitoring? ',
 'Parent Company',
 'Parent Ownership %']

In [8]:
ghg["FRS Id"] = (
    ghg["FRS Id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

violations_fac_year["REGISTRY_ID"] = (
    violations_fac_year["REGISTRY_ID"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

In [10]:
ghg_caa_violations = ghg.merge(violations_fac_year[["REGISTRY_ID", "YEAR", "ANNUAL_CAA_VIOLATION_COUNT"]], left_on = "FRS Id", right_on="REGISTRY_ID", how="left", suffixes=("_ghg_1", "_violations"))

In [11]:
ghg_caa_violations

,Facility Id,Facility Name,FRS Id,City,State,Zip Code,Latest Reporting Year,Address,County,Latitude,...,Is_Direct_Emitter,Is_Supplier,Basin,State where Emissions Occur,Does the facility employ continuous emissions monitoring?,Parent Company,Parent Ownership %,REGISTRY_ID,YEAR,ANNUAL_CAA_VIOLATION_COUNT
0,1001607,A B Paterson,110001245162,NEW ORLEANS,LA,70126,2010,5400 DWYER RD,Orleans,30.015800,...,True,False,NaN,NaN,NaN,Entergy Corporation,100.0,<NA>,NaN,NaN
1,1001221,AES Thames,110000316015,UNCASVILLE,CT,6382,2010,141 DEPOT ROAD,New London,41.428200,...,True,False,NaN,NaN,NaN,AES Corporation,100.0,<NA>,NaN,NaN
2,1008008,"AES Western Power, LLC",110001867551,PASADENA,TX,77506,2010,901 LIGHT COMAPNY ROAD,Harris,29.723900,...,True,False,NaN,NaN,NaN,The AES Corporation,100.0,<NA>,NaN,NaN
3,1002122,"AIR PRODUCTS & CHEMICALS, INC, Tesoro Martinez",110038091356,MARTINEZ,CA,94553,2010,NaN,CONTRA COSTA,38.026667,...,True,False,NaN,NaN,NaN,"Air Products and Chemicals, Inc.",100.0,<NA>,NaN,NaN
4,1003010,AIR PRODUCTS CHEMICALS INC,110043813246,GEISMAR,LA,70734,2010,36637 HWY 30,ASCENSION,30.205780,...,True,False,NaN,NaN,NaN,"Air Products and Chemicals, Inc.",100.0,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18175,1012745,"Rio Energy International, Inc.",<NA>,Houston,TX,77057,2023,5718 Westheimer Road Suite 1806,NaN,29.750397,...,False,True,NaN,NaN,NaN,RIO ENERGY INTERNATIONAL INC,100.0,<NA>,NaN,NaN
18176,1010465,SABIC Innovative Plastics US LLC,<NA>,Houston,TX,77042,2023,"2500 CityWest Blvd, Suite 100",NaN,42.462500,...,False,True,NaN,NaN,NaN,SABIC US HOLDINGS LP,100.0,<NA>,NaN,NaN
18177,1010713,Shell Chemical LP,110044228608,Houston,TX,77079,2023,150 North Dairy Ashford Road,HARRIS COUNTY,29.758090,...,False,True,NaN,NaN,NaN,SHELL PETROLEUM INC,100.0,<NA>,NaN,NaN
18178,1000621,THE LUBRIZOL CORPORATION,<NA>,WICKLIFFE,OH,44092,2023,29400 LAKELAND BLVD.,LAKE,41.613070,...,False,True,NaN,NaN,NaN,BERKSHIRE HATHAWAY INC,100.0,<NA>,NaN,NaN


In [13]:
#This is a quick data check to confirm the merge is working
ghg_caa_violations["ANNUAL_CAA_VIOLATION_COUNT"].sum()

np.float64(12247.0)

#Now let's merge GHG metadata with CAA penalties (sums and counts)

In [14]:
caa_penalties = pandas.read_csv("/content/drive/Shareddrives/EDGI - Shared NEW/12_Environmental_Enforcement_Watch/09_Data/Climate_Data_Project/ECHO data by FRSID/ECHO-Cross-Programs_GHGRP.ipynb outputs/ghgrp_fac_caa_penalties_2016-2023.csv", dtype = {"REGISTRY_ID": "string"})

In [15]:
caa_penalties.columns

Index(['PGM_SYS_ID', 'ACTIVITY_ID', 'ENF_IDENTIFIER', 'ACTIVITY_TYPE_CODE',
       'ACTIVITY_TYPE_DESC', 'STATE_EPA_FLAG', 'ENF_TYPE_CODE',
       'ENF_TYPE_DESC', 'SETTLEMENT_ENTERED_DATE', 'PENALTY_AMOUNT',
       'REGISTRY_ID', 'FAC_NAME', 'FAC_STREET', 'FAC_CITY', 'FAC_STATE',
       'FAC_ZIP', 'FAC_COUNTY', 'FAC_EPA_REGION', 'FAC_LAT', 'FAC_LONG',
       'FAC_DERIVED_WBD', 'FAC_DERIVED_CD113', 'FAC_PERCENT_MINORITY',
       'FAC_POP_DEN', 'FAC_DERIVED_HUC', 'FAC_SIC_CODES', 'FAC_NAICS_CODES',
       'DFR_URL'],
      dtype='object')

In [16]:
caa_penalties["YEAR"] = pandas.to_datetime(caa_penalties["SETTLEMENT_ENTERED_DATE"]).dt.year



In [17]:
penalties_fac_year = (
    caa_penalties[caa_penalties["YEAR"].between(2016, 2023)]
    .groupby(["REGISTRY_ID", "YEAR"], as_index=False)
    .agg(
        ANNUAL_CAA_PENALTY_TOTAL=("PENALTY_AMOUNT", "sum"),
        ANNUAL_CAA_PENALTY_CASES=("PENALTY_AMOUNT", "size")
    )
)

In [18]:
penalties_fac_year["REGISTRY_ID"] = (
    violations_fac_year["REGISTRY_ID"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

In [19]:
ghg_caa_penalties = ghg.merge(penalties_fac_year, left_on = "FRS Id", right_on="REGISTRY_ID", how="left", suffixes=("_ghg_2", "_penalties"))

In [22]:
#This is a quick data check to confirm the merge is working
ghg_caa_penalties["ANNUAL_CAA_PENALTY_CASES"].sum()

np.float64(7173.0)

In [23]:
ghg_caa_penalties.columns

Index(['Facility Id', 'Facility Name', 'FRS Id', 'City', 'State', 'Zip Code',
       'Latest Reporting Year', 'Address', 'County', 'Latitude', 'Longitude',
       'Primary NAICS Code', 'Industry Type (subparts)',
       'Industry Type (sectors)', 'Is_Direct_Emitter', 'Is_Supplier', 'Basin',
       'State where Emissions Occur',
       'Does the facility employ continuous emissions monitoring? ',
       'Parent Company', 'Parent Ownership %', 'REGISTRY_ID', 'YEAR',
       'ANNUAL_CAA_PENALTY_TOTAL', 'ANNUAL_CAA_PENALTY_CASES'],
      dtype='object')

#Finally, merge CAA violations and penalties per year to our GHG emissions aggregated by facility separately

In [24]:
#load the ghg aggregated emissions by facility data
ghg_emissions = pandas.read_csv("https://raw.githubusercontent.com/edgi-govdata-archiving/GHG-CDP/refs/heads/main/output_files/aggregated_emissions_by_facility_with_metadata.csv", dtype = {"FRS Id": "string"})

In [25]:
ghg_emissions.columns

Index(['Direct Point Emitters', 'Facility Id',
       'GHG Quantity Associated with CO2 Supply ',
       'GHG Quantity Associated with Coal-based liquid fuel production',
       'GHG Quantity Associated with Natural Gas Liquids Supply',
       'GHG Quantity Associated with Natural Gas Supply',
       'GHG Quantity Associated with Petroleum Products Exported',
       'GHG Quantity Associated with Petroleum Products Imported',
       'GHG Quantity Associated with Petroleum Products Produced',
       'Gathering & Boosting', 'LDC - Direct Emissions',
       'Onshore Oil & Gas Prod.', 'SF6 from Elec. Equip.',
       'Transmission Pipelines', 'Year', 'Total Direct Emissions',
       'Total Supplier Emissions', 'Unnamed: 0.2', 'Unnamed: 0.1',
       'Unnamed: 0', 'FRS Id', 'Facility Name', 'Address', 'City', 'State',
       'Zip Code', 'County', 'Latitude', 'Longitude', 'Primary NAICS Code',
       'Latest Reported Industry Type (subparts)',
       'Latest Reported Industry Type (sectors)', '

In [26]:
ghg_emissions["Facility Id"] = (
    ghg_emissions["Facility Id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

ghg_caa_violations["Facility Id"] = (
    ghg_caa_violations["Facility Id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

In [29]:
ghg_caa_violations_emissions = ghg_emissions.merge(
    ghg_caa_violations[["Facility Id", "YEAR", "ANNUAL_CAA_VIOLATION_COUNT"]],
    left_on=["Facility Id", "Year"], right_on=["Facility Id", "YEAR"], #note that we merged on violation year, not the penalty "SETTLEMENT_ENTERED_DATE"
    how="left", suffixes=("_echo_records", "_ghg_emissions")
)
ghg_caa_violations_emissions

,Direct Point Emitters,Facility Id,GHG Quantity Associated with CO2 Supply,GHG Quantity Associated with Coal-based liquid fuel production,GHG Quantity Associated with Natural Gas Liquids Supply,GHG Quantity Associated with Natural Gas Supply,GHG Quantity Associated with Petroleum Products Exported,GHG Quantity Associated with Petroleum Products Imported,GHG Quantity Associated with Petroleum Products Produced,Gathering & Boosting,...,CD_FIPS,DISTRICTID,Representative_Name,Representative_Last_Name,Representative_Party,Effective State,Parent Company,Parent Ownership %,YEAR,ANNUAL_CAA_VIOLATION_COUNT
0,293290.944,1000001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,5302.0,Rick Larsen,Larsen,Democrat,WA,"Empeco IV, LLC and USPF II Ferndale Holdings, ...",74.33298; 14.00002; 11.667,NaN,NaN
1,108094.104,1000002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,1803.0,Marlin A. Stutzman,Stutzman,Republican,IN,Saint-Gobain Containers Inc.,100.0,NaN,NaN
2,78408.200,1000003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,3701.0,Donald G. Davis,Davis,Democrat,NC,Saint-Gobain Containers Inc.,100.0,NaN,NaN
3,62346.096,1000004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.0,1715.0,Mary E. Miller,Miller,Republican,IL,Saint-Gobain Containers Inc.,100.0,NaN,NaN
4,74196.872,1000005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.0,613.0,Adam Gray,Gray,Democrat,CA,Saint-Gobain Containers Inc.,100.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113689,6485.500,1014921,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,5305.0,Michael Baumgartner,Baumgartner,Republican,WA,AVISTA CORP,100.0,NaN,NaN
113690,38971.804,1015119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.0,1311.0,Barry Loudermilk,Loudermilk,Republican,GA,VMC Specialty Alloys LLC,100.0,NaN,NaN
113691,NaN,1015125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,...,0.0,3800.0,Julie Fedorchak,Fedorchak,Republican,ND,Harvestone Low Carbon Partners,100.0,NaN,NaN
113692,37678.504,1015127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,2204.0,Mike Johnson,Johnson,Republican,LA,KRONOSPAN INC,100.0,NaN,NaN


In [30]:
ghg_emissions["Facility Id"] = (
    ghg_emissions["Facility Id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

ghg_caa_penalties["Facility Id"] = (
    ghg_caa_penalties["Facility Id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

In [33]:
ghg_caa_violations_penalties_emissions = ghg_caa_violations_emissions.merge(
    ghg_caa_penalties[["Facility Id", "YEAR","ANNUAL_CAA_PENALTY_TOTAL", "ANNUAL_CAA_PENALTY_CASES"]],
    left_on=["Facility Id", "Year"], right_on=["Facility Id", "YEAR"],
    how="left", suffixes=("_violations", "_penalties")
)
ghg_caa_violations_penalties_emissions

,Direct Point Emitters,Facility Id,GHG Quantity Associated with CO2 Supply,GHG Quantity Associated with Coal-based liquid fuel production,GHG Quantity Associated with Natural Gas Liquids Supply,GHG Quantity Associated with Natural Gas Supply,GHG Quantity Associated with Petroleum Products Exported,GHG Quantity Associated with Petroleum Products Imported,GHG Quantity Associated with Petroleum Products Produced,Gathering & Boosting,...,Representative_Last_Name,Representative_Party,Effective State,Parent Company,Parent Ownership %,YEAR_violations,ANNUAL_CAA_VIOLATION_COUNT,YEAR_penalties,ANNUAL_CAA_PENALTY_TOTAL,ANNUAL_CAA_PENALTY_CASES
0,293290.944,1000001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Larsen,Democrat,WA,"Empeco IV, LLC and USPF II Ferndale Holdings, ...",74.33298; 14.00002; 11.667,NaN,NaN,NaN,NaN,NaN
1,108094.104,1000002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Stutzman,Republican,IN,Saint-Gobain Containers Inc.,100.0,NaN,NaN,NaN,NaN,NaN
2,78408.200,1000003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Davis,Democrat,NC,Saint-Gobain Containers Inc.,100.0,NaN,NaN,NaN,NaN,NaN
3,62346.096,1000004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Miller,Republican,IL,Saint-Gobain Containers Inc.,100.0,NaN,NaN,NaN,NaN,NaN
4,74196.872,1000005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Gray,Democrat,CA,Saint-Gobain Containers Inc.,100.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114040,6485.500,1014921,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Baumgartner,Republican,WA,AVISTA CORP,100.0,NaN,NaN,NaN,NaN,NaN
114041,38971.804,1015119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Loudermilk,Republican,GA,VMC Specialty Alloys LLC,100.0,NaN,NaN,NaN,NaN,NaN
114042,NaN,1015125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,...,Fedorchak,Republican,ND,Harvestone Low Carbon Partners,100.0,NaN,NaN,NaN,NaN,NaN
114043,37678.504,1015127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Johnson,Republican,LA,KRONOSPAN INC,100.0,NaN,NaN,NaN,NaN,NaN


In [34]:
ghg_caa_violations_penalties_emissions.to_csv("ghg_caa_violations_penalties_emissions.csv")